# Tracing Executions

이 장에서는 로깅과 대화형 디버깅의 전제 조건인 실행 중 프로그램 상태를 관찰하는 방법을 보여줍니다. 파이썬의 강력한 기능 덕분에 몇 줄의 코드로 이를 구현할 수 있습니다.

In [197]:
import bookutils.setup
from bookutils import quiz
import Intro_Debugging

## Tracing Python Programs


디버깅 도구는 프로그램 실행 중에 상태에 접근하는 방법이 있습니다. 특히 파이썬 같은 _인터프리터_ 언어에서는 이 작업이 비교적 간단합니다. 언어가 인터프리터 방식이라면, 실행을 제어하고 상태를 검사하는 것이 일반적으로 쉽습니다. 왜냐하면 인터프리터가 이미 이러한 작업을 수행하고 있기 때문입니다. 디버거는 실행을 중단하고 프로그램 상태에 접근할 수 있는 훅을 기반으로 구현됩니다.

Python은 sys.settrace() 함수에서 이러한 훅을 제공합니다. 이 함수는 실행되는 모든 줄에서 호출될 추적 함수와 함께 호출됩니다. 예를 들어:

```python
sys.settrace(traceit)
```

이러한 추적 함수는 매우 편리한데, 이는 단순히 모든 것_을 추적하기 때문입니다. 대화형 디버거와는 달리, 실행의 어떤 측면에 관심이 있는지를 선택할 필요 없이, 긴 추적을 실행 로그로 출력하여 나중에 검토할 수 있습니다.
이 추적 함수는 다음과 같은 형식을 가집니다.

In [198]:
def traceit(frame, event, arg):
    ...

Here, `event` is a string telling what has happened in the program – for instance,

* `'line'` – a new line is executed
* `'call'` – a function just has been called
* `'return'` – a function returns

The `frame` argument holds the current execution frame – that is, the function and its local variables:

* `frame.f_lineno` – the current line
* `frame.f_locals` – the current variables (as a Python dictionary)
* `frame.f_code` – the current code (as a Code object), with attributes such as
    * `frame.f_code.co_name` – the name of the current function

In [199]:
def traceit(frame, event, arg):
    print(event, frame.f_lineno, frame.f_code.co_name, frame.f_locals)

trace 함수는 감시하는 함수의 다음 이벤트에 실행될 것을 반환하므로 자기 자신을 반환한다.

In [200]:
def traceit(frame, event, arg):
    print(event, frame.f_lineno, frame.f_code.co_name, frame.f_locals)
    return traceit

In [201]:
from Intro_Debugging import remove_html_markup
import inspect
from bookutils import print_content
import sys

```python
237  def remove_html_markup(s):  # type: ignore
238      tag = False
239      quote = False
240      out = ""
241  
242      for c in s:
243          assert tag or not quote
244  
245          if c == '<' and not quote:
246              tag = True
247          elif c == '>' and not quote:
248              tag = False
249          elif (c == '"' or c == "'") and tag:
250              quote = not quote
251          elif not tag:
252              out = out + c
253  
254      return out
```

추적을 켜고, `remove_html_markup()`을 호출한 다음, 추적을 다시 끄는 `remove_html_markup_traced()`을 정의합니다.

In [202]:
def remove_html_markup_traced(s): # type: ignore
    sys.settrace(traceit)          # 추적 시작
    ret = remove_html_markup(s)
    sys.settrace(None)             # 추적 중단
    return ret

`remove_html_markup_traced()`를 실행하면 다음과 같은 결과를 얻습니다:
* 먼저 `call` 이벤트를 얻습니다 (`remove_html_markup()` 호출을 보여줌)
* 그 다음 여러 `line` 이벤트를 얻습니다 (`remove_html_markup()`의 각 라인에 대해)
* 마지막으로 `return` 이벤트를 얻습니다 (`remove_html_markup()`에서 반환을 보여줌)

In [203]:
remove_html_markup_traced('xyz')

call 237 remove_html_markup {'s': 'xyz'}
line 238 remove_html_markup {'s': 'xyz'}
line 239 remove_html_markup {'s': 'xyz', 'tag': False}
line 240 remove_html_markup {'s': 'xyz', 'tag': False, 'quote': False}
line 242 remove_html_markup {'s': 'xyz', 'tag': False, 'quote': False, 'out': ''}
line 243 remove_html_markup {'s': 'xyz', 'tag': False, 'quote': False, 'out': '', 'c': 'x'}
line 245 remove_html_markup {'s': 'xyz', 'tag': False, 'quote': False, 'out': '', 'c': 'x'}
line 247 remove_html_markup {'s': 'xyz', 'tag': False, 'quote': False, 'out': '', 'c': 'x'}
line 249 remove_html_markup {'s': 'xyz', 'tag': False, 'quote': False, 'out': '', 'c': 'x'}
line 251 remove_html_markup {'s': 'xyz', 'tag': False, 'quote': False, 'out': '', 'c': 'x'}
line 252 remove_html_markup {'s': 'xyz', 'tag': False, 'quote': False, 'out': '', 'c': 'x'}
line 242 remove_html_markup {'s': 'xyz', 'tag': False, 'quote': False, 'out': 'x', 'c': 'x'}
line 243 remove_html_markup {'s': 'xyz', 'tag': False, 'quote': F

'xyz'

흥미로운 점은 이러한 모든 지역 변수들을 일반 Python 객체처럼 실제로 접근할 수 있다는 것입니다. 예를 들어, `frame.f_locals['c']`를 통해 `c`의 값에 개별적으로 접근할 수 있습니다:

In [204]:
def traceit(frame, event, arg):
    # 지역 변수 c만 출력하도록 수정
    if 'c' in frame.f_locals: 
        value_of_c = frame.f_locals['c']
        print(f"{frame.f_lineno:} c = {repr(value_of_c)}")
    else:
        print(f"{frame.f_lineno:} c is undefined")

    return traceit

In [205]:
remove_html_markup_traced('xyz')

237 c is undefined
238 c is undefined
239 c is undefined
240 c is undefined
242 c is undefined
243 c = 'x'
245 c = 'x'
247 c = 'x'
249 c = 'x'
251 c = 'x'
252 c = 'x'
242 c = 'x'
243 c = 'y'
245 c = 'y'
247 c = 'y'
249 c = 'y'
251 c = 'y'
252 c = 'y'
242 c = 'y'
243 c = 'z'
245 c = 'z'
247 c = 'z'
249 c = 'z'
251 c = 'z'
252 c = 'z'
242 c = 'z'
254 c = 'z'
254 c = 'z'


'xyz'

`sys.settrace()`가 `None`을 반환하면 현재 스코프에서 추적이 중단됩니다; 추적은 현재 함수가 반환될 때 다시 시작됩니다.

## A Tracer Class

추적 함수를 좀 더 개선해 보겠습니다. 먼저, 필요에 따라 실제로 추적을 사용자화 할 수 있다면 좋을 것 같습니다. 이를 위해 모든 포맷팅을 처리하고 다양한 출력 형식을 허용하기 위해 상속될 수 있는 `Tracer` 클래스를 도입합니다.

`Tracer`의 `traceit()` method는 위와 동일하며, 다시 `sys.settrace()`를 통해 설정됩니다. Python `print()` function 대신 `log()` method를 사용합니다.

In [206]:
from StackInspector import StackInspector

In [207]:
class Tracer(StackInspector):
    def __init__(self, *, file = sys.stdout):
        self.original_trace_function = None
        self.file = file


    def traceit(self, frame, event, arg):
        self.log(event, frame.f_lineno, frame.f_code.co_name, frame.f_locals)


    def _traceit(self, frame, event, arg):
        if self.our_frame(frame):
            pass
        else:
            # 추적한 함수의 정보를 출력하는 부분
            self.traceit(frame, event, arg)
        
        # 마찬가지로, 자기 자신을 반환하여 추적을 계속합니다.
        return self._traceit


    def log(self, *objects, sep = ' ', end = '\n', flush = True):
        print(*objects, sep=sep, end=end, file=self.file, flush=flush)


    def __enter__(self):
        self.original_trace_function = sys.gettrace()
        sys.settrace(self._traceit)
        return self


    def __exit__(self, exc_tp, exc_value, exc_traceback):
        sys.settrace(self.original_trace_function)
        if self.is_internal_error(exc_tp, exc_value, exc_traceback):
            return False
        else:
            return None

`Tracer` 클래스를 사용하는 방법입니다. 이전과 동일하게 작동하지만 사용하기가 더 편리해졌습니다:

In [208]:
with Tracer():
    remove_html_markup("abc")

call 237 remove_html_markup {'s': 'abc'}
line 238 remove_html_markup {'s': 'abc'}
line 239 remove_html_markup {'s': 'abc', 'tag': False}
line 240 remove_html_markup {'s': 'abc', 'tag': False, 'quote': False}
line 242 remove_html_markup {'s': 'abc', 'tag': False, 'quote': False, 'out': ''}
line 243 remove_html_markup {'s': 'abc', 'tag': False, 'quote': False, 'out': '', 'c': 'a'}
line 245 remove_html_markup {'s': 'abc', 'tag': False, 'quote': False, 'out': '', 'c': 'a'}
line 247 remove_html_markup {'s': 'abc', 'tag': False, 'quote': False, 'out': '', 'c': 'a'}
line 249 remove_html_markup {'s': 'abc', 'tag': False, 'quote': False, 'out': '', 'c': 'a'}
line 251 remove_html_markup {'s': 'abc', 'tag': False, 'quote': False, 'out': '', 'c': 'a'}
line 252 remove_html_markup {'s': 'abc', 'tag': False, 'quote': False, 'out': '', 'c': 'a'}
line 242 remove_html_markup {'s': 'abc', 'tag': False, 'quote': False, 'out': 'a', 'c': 'a'}
line 243 remove_html_markup {'s': 'abc', 'tag': False, 'quote': F

## Accessing Source Code

이제 class를 추가 기능으로 _extend_ 할 수 있습니다. 추적 중인 함수의 source code를 실제로 표시할 수 있다면 좋을 것 같습니다. 이를 통해 우리가 어디에 있는지 알 수 있죠. Python에서 `inspect.getsource()` 함수는 함수나 module의 source code를 반환합니다. 다음과 같이 조회하면
 
```python
module = inspect.getmodule(frame.f_code)
```
 
현재 module을 얻을 수 있고,
 
```python
inspect.getsource(module)
```
 
이를 통해 source code를 얻을 수 있습니다. 이제 현재 line만 가져오면 됩니다.

확장된 `traceit()` 메서드를 구현하기 위해 약간의 해킹 기법을 사용합니다. Python 언어는 모든 메서드가 하나의 연속된 단위로 정의된 전체 클래스를 요구하지만, 우리는 메서드를 하나씩 도입하고 싶습니다. 이 문제를 해결하기 위해 특별한 해킹 기법을 사용합니다: 클래스 `C`에 새로운 메서드를 도입하고 싶을 때마다 다음과 같은 구조를 사용합니다

```python
class C(C):
    def new_method(self, args):
        pass
```

이것은 `C`를 자기 자신의 서브클래스로 정의하는 것처럼 보이는데, 이는 말이 안되는 것 같습니다 - 하지만 실제로는 _이전_ `C` 클래스의 서브클래스로 새로운 `C` 클래스를 도입하고, 이전 `C` 정의를 가리는 것입니다. 이를 통해 우리가 원하는 대로 `new_method()`를 메서드로 가지는 `C` 클래스를 얻을 수 있습니다. (단, 이전에 정의된 `C` 객체들은 이전 `C` 정의를 유지하므로 다시 생성해야 합니다.)

In [209]:
class Tracer(Tracer):
    def traceit(self, frame, event, arg):
        if event == 'line':
            module = inspect.getmodule(frame.f_code)
            if module is None:
                source = inspect.getsource(frame.f_code)
            else:
                source = inspect.getsource(module)
            current_line = source.split('\n')[frame.f_lineno - 1]
            self.log(frame.f_lineno, current_line)

In [210]:
with Tracer():
    remove_html_markup("abc")

238     tag = False
239     quote = False
240     out = ""
242     for c in s:
243         assert tag or not quote
245         if c == '<' and not quote:
247         elif c == '>' and not quote:
249         elif (c == '"' or c == "'") and tag:
251         elif not tag:
252             out = out + c
242     for c in s:
243         assert tag or not quote
245         if c == '<' and not quote:
247         elif c == '>' and not quote:
249         elif (c == '"' or c == "'") and tag:
251         elif not tag:
252             out = out + c
242     for c in s:
243         assert tag or not quote
245         if c == '<' and not quote:
247         elif c == '>' and not quote:
249         elif (c == '"' or c == "'") and tag:
251         elif not tag:
252             out = out + c
242     for c in s:
254     return out


## Tracing Calls and Returns

다음으로, 함수의 호출과 반환을 보고하고 싶습니다. `return` 이벤트의 경우, `arg`는 반환되는 값을 가지고 있습니다.

In [211]:
class Tracer(Tracer):
    def traceit(self, frame, event, arg):
        if event == 'call':
            self.log(f"Calling {frame.f_code.co_name}()")

        if event == 'line':
            module = inspect.getmodule(frame.f_code)
            if module:
                source = inspect.getsource(module)
            if source:
                current_line = source.split('\n')[frame.f_lineno - 1]
                self.log(frame.f_lineno, current_line)

        if event == 'return':
            self.log(f"{frame.f_code.co_name}() returns {repr(arg)}")

In [212]:
with Tracer():
    remove_html_markup("abc")

Calling remove_html_markup()
238     tag = False
239     quote = False
240     out = ""
242     for c in s:
243         assert tag or not quote
245         if c == '<' and not quote:
247         elif c == '>' and not quote:
249         elif (c == '"' or c == "'") and tag:
251         elif not tag:
252             out = out + c
242     for c in s:
243         assert tag or not quote
245         if c == '<' and not quote:
247         elif c == '>' and not quote:
249         elif (c == '"' or c == "'") and tag:
251         elif not tag:
252             out = out + c
242     for c in s:
243         assert tag or not quote
245         if c == '<' and not quote:
247         elif c == '>' and not quote:
249         elif (c == '"' or c == "'") and tag:
251         elif not tag:
252             out = out + c
242     for c in s:
254     return out
remove_html_markup() returns 'abc'


## Tracing Variable Changes

현재 라인이 실행되면서 이전 라인과 달라진(혹은 새로 추가된) 변수를 report하고 싶습니다.

In [213]:
class Tracer(Tracer):
    def __init__(self, file = sys.stdout):
        self.last_vars = {} # 값이 변경된 변수를 저장할 딕셔너리
        super().__init__(file=file)

    def changed_vars(self, new_vars):
        changed = {}
        # last_vars에 없던 변수 혹은 있지만 값이 다른 것을 새로운 딕셔너리에 저장
        for var_name, var_value in new_vars.items():
            if (var_name not in self.last_vars or
                    self.last_vars[var_name] != var_value):
                changed[var_name] = var_value
        # last_vars를 업데이트
        self.last_vars = new_vars.copy()
        # 변경된 변수들을 반환
        return changed

Here's how this works: If variable `a` is set to 10 (and we didn't have it so far), it is marked as changed:

In [214]:
tracer = Tracer()

In [215]:
# a라는 변수가 새로 추가된 상황
tracer.changed_vars({'a': 10})

{'a': 10}

In [216]:
# b라는 변수가 새로 추가된 상황
tracer.changed_vars({'a': 10, 'b': 25})

{'b': 25}

In [217]:
# a,b 모두 이전 라인과 동일한 값을 갖는 상황
tracer.changed_vars({'a': 10, 'b': 25})

{}

In [218]:
# a,b가 삭제되고 c,d가 새로 추가된 상황
changes = tracer.changed_vars({'c': 10, 'd': 25})
changes

{'c': 10, 'd': 25}

In [219]:
", ".join([var + " = " + repr(changes[var]) for var in changes])

'c = 10, d = 25'

이제 모든 것을 추적 함수에 모아서, 변수의 변화를 확인할 수 있습니다. 함수 호출 시에는 모든 변수가 "새로운" 값을 가지고, 함수에서 반환할 때는 명시적으로 "마지막" 변수들을 삭제한다는 점을 활용합니다.

In [220]:
class Tracer(Tracer):
    def print_debugger_status(self, frame, event, arg):
        """Show current source line and changed vars"""
        changes = self.changed_vars(frame.f_locals)
        changes_s = ", ".join([var + " = " + repr(changes[var])
                               for var in changes])

        if event == 'call':
            self.log("Calling " + frame.f_code.co_name + '(' + changes_s + ')')
        elif changes:
            # 변화한 변수가 있는 경우 출력
            self.log(' ' * 40, '#', changes_s)

        if event == 'line':
            try:
                module = inspect.getmodule(frame.f_code)
                if module is None:
                    source = inspect.getsource(frame.f_code)
                else:
                    source = inspect.getsource(module)
                current_line = source.split('\n')[frame.f_lineno - 1]

            except OSError as err:
                self.log(f"{err.__class__.__name__}: {err}")
                current_line = ""

            self.log(repr(frame.f_lineno) + ' ' + current_line)

        if event == 'return':
            self.log(frame.f_code.co_name + '()' + " returns " + repr(arg))
            self.last_vars = {}  # Delete 'last' variables

    def traceit(self, frame, event, arg):
        """Tracing function; called at every line. To be overloaded in subclasses."""
        self.print_debugger_status(frame, event, arg)

In [221]:
with Tracer():
    remove_html_markup('<b>x</b>')

Calling remove_html_markup(s = '<b>x</b>')
238     tag = False
                                         # tag = False
239     quote = False
                                         # quote = False
240     out = ""
                                         # out = ''
242     for c in s:
                                         # c = '<'
243         assert tag or not quote
245         if c == '<' and not quote:
246             tag = True
                                         # tag = True
242     for c in s:
                                         # c = 'b'
243         assert tag or not quote
245         if c == '<' and not quote:
247         elif c == '>' and not quote:
249         elif (c == '"' or c == "'") and tag:
251         elif not tag:
242     for c in s:
                                         # c = '>'
243         assert tag or not quote
245         if c == '<' and not quote:
247         elif c == '>' and not quote:
248             tag = False
                              

## Conditional Tracing

위와 같은 로그는 실행 시간이 오래 걸리거나 데이터 구조가 매우 복잡해지면 지저분해질 수 있습니다. 예를 들어 로컬 변수 중 하나가 1,000개의 항목이 있는 리스트이고 각 라인마다 변경된다면, 매 단계마다 1,000개의 항목이 있는 전체 리스트를 출력하게 될 것입니다.

더 나은 대안은 특정 조건이 충족될 때만 로그를 기록하도록 트레이서를 설정하는 것입니다.

이를 위해 실행 중에 확인할 condition 을 받는 `ConditionalTracer` 클래스를 도입합니다. 이 조건이 충족될 때만 현재 상태를 나열합니다. 다음과 같이 사용할 수 있습니다:

```python
with ConditionalTracer(condition='c == "z"'):
    remove_html_markup(...)
```

위 코드는 `c`가 `'z'` 값을 가질 때만 실행되고, 다음과 같이 사용할 수 있습니다:

```python
with ConditionalTracer(condition='quote'):
    remove_html_markup(...)
```

`quote`가 True인 동안 실행된 라인만 얻을 수 있습니다. 여러 조건이 있는 경우 `and`, `or`, `not`을 사용하여 하나로 결합할 수 있습니다.

In [222]:
class ConditionalTracer(Tracer):
    def __init__(self, *, condition = None, file = sys.stdout):
        if condition is None:
            condition = 'False'
        assert isinstance(condition, str)

        self.condition = condition
        self.last_report = None
        super().__init__(file=file)

`traceit()` 함수는 `condition`을 평가하고 조건이 충족될 때만 현재 라인을 보고합니다. 이를 위해 Python의 `eval()` 함수를 사용하여 테스트 중인 프로그램의 로컬 변수 컨텍스트에서 조건을 평가합니다. 조건이 설정되면 경과 시간을 나타내기 위해 점 세 개를 출력합니다.

> eval(expression, globals=None, locals=None)


In [223]:
class ConditionalTracer(ConditionalTracer):
    def eval_in_context(self, expr, frame): # -> Optional[bool]
        try:
            # 현재 프레임의 지역변수에 대해 조건식을 판별함
            cond = eval(expr, None, frame.f_locals)
        except NameError:
            cond = None
        return cond

`do_report()` 함수는 상태가 보고되어야 하는 경우 True를 반환합니다:

In [224]:
class ConditionalTracer(ConditionalTracer):
    def do_report(self, frame, event, arg):
        return self.eval_in_context(self.condition, frame)

In [225]:
class ConditionalTracer(ConditionalTracer):
    def traceit(self, frame, event, arg):
        # condition을 만족하는 이벤트의 경우 report = True
        report = self.do_report(frame, event, arg)

        # report: false -> true로 바뀌면 ... 출력 (라인 번호 확인)
        if report != self.last_report:
            if report:
                self.log("...")
            self.last_report = report

        # report: true인 경우 현재 상태 출력
        if report:
            self.print_debugger_status(frame, event, arg)

Here's an example. We see that `quote` is set only while the three characters `b`, `a`, and `r` are processed (as should be).

In [226]:
with ConditionalTracer(condition='quote'):
    remove_html_markup('<b title="bar">"foo"</b>')
    # quote가 True인 동안 코드라인 출력

...
                                         # s = '<b title="bar">"foo"</b>', tag = True, quote = True, out = '', c = '"'
242     for c in s:
                                         # c = 'b'
243         assert tag or not quote
245         if c == '<' and not quote:
247         elif c == '>' and not quote:
249         elif (c == '"' or c == "'") and tag:
251         elif not tag:
242     for c in s:
                                         # c = 'a'
243         assert tag or not quote
245         if c == '<' and not quote:
247         elif c == '>' and not quote:
249         elif (c == '"' or c == "'") and tag:
251         elif not tag:
242     for c in s:
                                         # c = 'r'
243         assert tag or not quote
245         if c == '<' and not quote:
247         elif c == '>' and not quote:
249         elif (c == '"' or c == "'") and tag:
251         elif not tag:
242     for c in s:
                                         # c = '"'
243         assert t

특정 코드 위치에만 로그를 집중할 수도 있습니다. 이를 위해 `function`과 `line` 의사 변수를 평가 컨텍스트에 추가합니다. 이 변수들은 조건문 내에서 현재 함수 이름이나 라인을 참조하는 데 사용될 수 있습니다. 그런 다음 위와 같이 원래의 `eval_cond()`를 호출합니다.

In [227]:
class ConditionalTracer(ConditionalTracer):
    def eval_in_context(self, expr, frame):
        # 함수 이름이나 라인 번호는 변수에 없는 정보이므로 임의로 추가
        frame.f_locals['function'] = frame.f_code.co_name
        frame.f_locals['line'] = frame.f_lineno

        # 원래 위에서 정의했던 ConditionalTracer의 eval_in_context 함수 호출
        # ... eval(expr, None, frame.f_locals) ... 
        return super().eval_in_context(expr, frame)

In [228]:
with ConditionalTracer(condition="function == 'remove_html_markup' and line >= 237"):
    remove_html_markup('xyz')

...
Calling remove_html_markup(s = 'xyz', function = 'remove_html_markup', line = 237)
                                         # line = 238
238     tag = False
                                         # line = 239, tag = False
239     quote = False
                                         # line = 240, quote = False
240     out = ""
                                         # line = 242, out = ''
242     for c in s:
                                         # line = 243, c = 'x'
243         assert tag or not quote
                                         # line = 245
245         if c == '<' and not quote:
                                         # line = 247
247         elif c == '>' and not quote:
                                         # line = 249
249         elif (c == '"' or c == "'") and tag:
                                         # line = 251
251         elif not tag:
                                         # line = 252
252             out = out + c
                          

## Watching Events

변수가 특정 값을 가지고 있을 때뿐만 아니라 *그 값이 변경될 때*를 정확히 추적하는 데에도 관심이 있을 수 있습니다.

이를 위해 우리는 어떤 이벤트가 발생할 때를 감시하는 `EventTracer` 클래스를 설정합니다. 이 클래스는 변수들을 받아서 각 라인마다 평가하며, 어떤 값이 변경되면 상태를 기록합니다.

```python
with EventTracer(events=['tag', 'quote']):
    remove_html_markup(...)
```

예를 들어, `tag` 또는 `quote`의 값이 변경되는 모든 라인의 목록을 얻을 수 있습니다; 그리고

```python
with EventTracer(events=['function']):
    remove_html_markup(...)
```

현재 함수가 변경되는 모든 라인의 목록을 얻을 수 있습니다.

In [229]:
class EventTracer(ConditionalTracer):
    def __init__(self, *, condition = None, events = [], file = sys.stdout):
        assert isinstance(events, list)
        self.events = events
        self.last_event_values = {}
        super().__init__(file=file, condition=condition)

`events_changed()` 함수는 개별 이벤트들을 평가하고 그것들이 변경되었는지 확인합니다.

In [230]:
class EventTracer(EventTracer):
    def events_changed(self, events, frame):
        change = False
        for event in events:
            # 변수(혹은 이벤트)의 평가값을 frame 컨텍스트에서 가져옴
            value = self.eval_in_context(event, frame)

            # 새로 추가된 변수거나,
            # 이전에 평가한 값과 다른 경우 변경된 것으로 간주
            if (event not in self.last_event_values or
                    value != self.last_event_values[event]):
                self.last_event_values[event] = value
                change = True

        return change

In [231]:
class EventTracer(EventTracer):
    def do_report(self, frame, event, arg):
        return (self.eval_in_context(self.condition, frame) or self.events_changed(self.events, frame))

This allows us to track, for instance, how `quote` and `tag` change their values over time.

In [232]:
with EventTracer(events=['quote', 'tag']):
    remove_html_markup('<b title="bar">"foo"</b>')

...
Calling remove_html_markup(s = '<b title="bar">"foo"</b>', function = 'remove_html_markup', line = 237)
...
                                         # line = 239, tag = False
239     quote = False
                                         # line = 240, quote = False
240     out = ""
...
                                         # line = 242, tag = True, out = '', c = '<'
242     for c in s:
...
                                         # quote = True, c = '"'
242     for c in s:
...
                                         # quote = False
242     for c in s:
...
                                         # tag = False, c = '>'
242     for c in s:
...
                                         # tag = True, out = '"foo"', c = '<'
242     for c in s:
...
                                         # tag = False, c = '>'
242     for c in s:


## Efficient Tracing

우리의 프레임워크는 모든 조건과 이벤트를 프로그램의 각 라인에 대해 평가해야 하기 때문에 매우 느립니다.

In [233]:
from Timer import Timer

In [234]:
runs = 1000

In [235]:
with Timer() as t:
    for i in range(runs):
        remove_html_markup('<b title="bar">"foo"</b>')
untraced_execution_time = t.elapsed_time()
untraced_execution_time

0.0019432130002314807

And here's the _traced_ execution time:

In [236]:
with Timer() as t:
    for i in range(runs):
        with EventTracer():
            remove_html_markup('<b title="bar">"foo"</b>')
traced_execution_time = t.elapsed_time()
traced_execution_time

1.4893834099998458

우리는 추적된 실행 시간이 몇 백 배 느리다는 것을 알 수 있습니다:

In [237]:
traced_execution_time / untraced_execution_time

766.4540170441564

하지만 추적하면서도 프로그램을 최대 속도로 실행할 수 있는 트릭이 있습니다. 런타임에 조건을 동적으로 확인하는 대신, 테스트 중인 프로그램에 적절한 코드를 정적으로 삽입할 수 있습니다. 이렇게 하면 추적되지 않는 코드는 정상 속도로 실행됩니다.

하지만 단점이 있습니다: 이는 검사할 조건이 특정 위치로 제한되는 경우에만 작동합니다 - 우리가 추적 코드를 삽입하는 위치가 바로 그 위치이기 때문입니다. 하지만 이러한 제한에도 불구하고, 정적 추적은 실행 속도를 크게 향상시킬 수 있습니다.

정적 코드 삽입은 어떻게 작동할까요? 이 트릭은 주어진 위치에 특별한 디버깅 구문을 삽입하기 위해 프로그램 코드를 재작성하는 것을 포함합니다. 이렇게 하면 추적 함수를 전혀 사용할 필요가 없습니다.

다음의 `insert_tracer()` 함수가 이를 보여줍니다. 이 함수는 함수와 추적 구문을 삽입할 breakpoint 라인들의 리스트를 받습니다. 각각의 주어진 라인에서, 다음의 코드를 삽입합니다

In [238]:
TRACER_CODE = \
    "TRACER.print_debugger_status(inspect.currentframe(), 'line', None); "

함수 정의에 삽입되며, 이는 다음의 tracer를 호출합니다:

In [239]:
TRACER = Tracer()

그런 다음 `insert_tracer()`는 결과 코드를 새로운 "추적된" 함수로 컴파일하고, 이를 반환합니다.

In [240]:
def insert_tracer(function, breakpoints = []):
    # 중단점을 추가할 함수의 코드라인을 불러옴
    source_lines, starting_line_number = inspect.getsourcelines(function)

    # 중단점을 역순으로 삽입
    breakpoints.sort(reverse=True)
    for given_line in breakpoints:
        # 중단점을 원래 코드에 삽입
        relative_line = given_line - starting_line_number + 1
        inject_line = source_lines[relative_line - 1]
        indent = len(inject_line) - len(inject_line.lstrip())
        source_lines[relative_line - 1] = ' ' * indent + TRACER_CODE + inject_line.lstrip()

    # 함수 이름 변경
    new_function_name = function.__name__ + "_traced"
    source_lines[0] = source_lines[0].replace(function.__name__, new_function_name)
    new_def = "".join(source_lines)

    print_content(new_def, '.py', start_line_number=starting_line_number)

    # 디버깅을 용이하게 하기 위해 원래 소스와 파일 이름을 유지
    prefix = '\n' * starting_line_number    # 줄 번호 조정
    new_function_code = compile(prefix + new_def, function.__code__.co_filename, 'exec')
    exec(new_function_code)
    new_function = eval(new_function_name)
    return new_function

예시: `remove_html_markup()`의 (상대적) 7번째와 18번째 줄에 두 개의 중단점을 삽입하면 다음과 같이 `remove_html_markup_traced()`의 정의가 재작성됩니다:

In [241]:
_, remove_html_markup_starting_line_number = inspect.getsourcelines(remove_html_markup)
breakpoints = [(remove_html_markup_starting_line_number - 1) + 7, 
               (remove_html_markup_starting_line_number - 1) + 18]

In [242]:
remove_html_markup_traced = insert_tracer(remove_html_markup, breakpoints)

237  def remove_html_markup_traced(s):  # type: ignore
238      tag = False
239      quote = False
240      out = ""
241  
242      for c in s:
243          TRACER.print_debugger_status(inspect.currentframe(), 'line', None); assert tag or not quote
244  
245          if c == '<' and not quote:
246              tag = True
247          elif c == '>' and not quote:
248              tag = False
249          elif (c == '"' or c == "'") and tag:
250              quote = not quote
251          elif not tag:
252              out = out + c
253  
254      TRACER.print_debugger_status(inspect.currentframe(), 'line', None); return out

정적으로 추적되는 `remove_html_markup_traced()`를 실행하면 동적 추적기를 사용할 때와 동일한 출력을 얻습니다. 참고로 나열된 소스 코드는 원본 코드를 보여주며, `TRACER`에 주입된 호출은 표시되지 않습니다.

In [243]:
with Timer() as t:
    remove_html_markup_traced('<b title="bar">"foo"</b>')
static_tracer_execution_time = t.elapsed_time()

                                         # s = '<b title="bar">"foo"</b>', tag = False, quote = False, out = '', c = '<'
244 
                                         # tag = True, c = 'b'
244 
                                         # c = ' '
244 
                                         # c = 't'
244 
                                         # c = 'i'
244 
                                         # c = 't'
244 
                                         # c = 'l'
244 
                                         # c = 'e'
244 
                                         # c = '='
244 
                                         # c = '"'
244 
                                         # quote = True, c = 'b'
244 
                                         # c = 'a'
244 
                                         # c = 'r'
244 
                                         # c = '"'
244 
                                         # quote = False, c = '>'
244 
                                         # tag = 

정적 추적기는 동적 추적기와 비교하여 얼마나 빠른가요? 다음은 위 코드의 실행 시간입니다:

In [244]:
static_tracer_execution_time

0.04150557100001606

이와 동등한 동적 추적기와 비교해보겠습니다:

In [245]:
line7 = (remove_html_markup_starting_line_number - 1) + 7
line18 = (remove_html_markup_starting_line_number - 1) + 18

with Timer() as t:
    with EventTracer(condition=f'line == {line7} or line == {line18}'):
        remove_html_markup('<b title="bar">"foo"</b>')

dynamic_tracer_execution_time = t.elapsed_time()
dynamic_tracer_execution_time

...
                                         # s = '<b title="bar">"foo"</b>', function = 'remove_html_markup', line = 243, tag = False, quote = False, out = '', c = '<'
243         assert tag or not quote
...
                                         # tag = True, c = 'b'
243         assert tag or not quote
...
                                         # c = ' '
243         assert tag or not quote
...
                                         # c = 't'
243         assert tag or not quote
...
                                         # c = 'i'
243         assert tag or not quote
...
                                         # c = 't'
243         assert tag or not quote
...
                                         # c = 'l'
243         assert tag or not quote
...
                                         # c = 'e'
243         assert tag or not quote
...
                                         # c = '='
243         assert tag or not quote
...
                                         # c = '"'

0.08101799400037635

In [246]:
dynamic_tracer_execution_time / static_tracer_execution_time

1.9519787837720628

실제로 함수를 한 줄씩 실행하는 것은 매우 비용이 많이 들 수 있습니다. `some_extreme_function()`과 `remove_html_markup()`의 모든 호출, 라인, 반환이 추적되기 때문입니다.

반면에 정적 추적기는 코드의 특정 위치를 참조하는 조건들로 제한됩니다. 예를 들어 어떤 변수가 변경되는지 확인하려면 변경이 가능한 위치를 결정하기 위해 (간단하지 않은) 정적 코드 분석을 수행해야 합니다. 만약 변수가 참조나 포인터를 통해 간접적으로 변경된다면(C와 같은 시스템 수준 언어에서 흔한 위험), 각 명령어 실행 후 실제로 그 값을 관찰하는 것 외에는 다른 대안이 없습니다.

## Exercises


### Exercise 1: Exception Handling

지금까지는 개별 함수의 라인 실행만을 살펴보았습니다. 하지만 함수가 예외를 발생시키는 경우, 이를 포착하고 보고하고 싶을 수 있습니다. 현재는 예외가 우리의 추적기를 통과하여 발생하면서 추적이 중단됩니다.

In [262]:
def fail():
    return 2 / 0

In [248]:
with Tracer():
    try:
        fail()
    except Exception:
        pass

Calling fail()
2     return 2 / 0
fail() raises ZeroDivisionError: division by zero
fail() returns None


`Tracer` 클래스(또는 `EventTracer` 서브클래스)를 확장하여 예외(이벤트 타입 `'exception'`)도 다음과 같이 적절히 추적되도록 하세요:

```
fail() raises ZeroDivisionError: division by zero
```

#### Solution
Simply extend `print_debugger_status()`:

In [249]:
class Tracer(Tracer):
    def print_debugger_status(self, frame, event, arg):
        if event == 'exception':
            exception, value, tb = arg
            self.log(f"{frame.f_code.co_name}() "
                     f"raises {exception.__name__}: {value}")
        else:
            super().print_debugger_status(frame, event, arg)

In [250]:
with Tracer():
    try:
        fail()
    except Exception:
        pass

Calling fail()
2     return 2 / 0
fail() raises ZeroDivisionError: division by zero
fail() returns None
